# Update user data from CSV

Enrich an existing GitHub users CSV (must have a **`Username`** column).

## Recommended: one API pass per user

Run **section 1 (unified)** — a single **GraphQL** query (paginated at 100 repos/page) fills every API column at once:

`Public_Repositories`, `Lifetime_Commits`, `Followers`, `Total_Stars`, `Repo_Names`, `Repo_Metadata`

That replaces sections 1 + 2a–2d, which would otherwise hit `/users/{login}` up to **twice** and `/users/{login}/repos` up to **three times** per user.

| Section | Calls per user (typical) | Columns |
|---------|--------------------------|---------|
| **1 unified** (recommended) | ~1–3 GraphQL pages | all API columns above |
| 1 legacy + 2a–2d | 2 REST + 4× REST repos + GraphQL | same (use only if you need one column) |
| 3 scrape | 1–2 HTTP (no token) | `Contributions_Last_Year`, `Heatmap_Data` |

**Token:** set `GITHUB_KEY=ghp_...` in `.env`. Section 3 uses polite delays, no token.

In [ ]:
# Optional: set a token here if the kernel does not see .env
# import os
# os.environ["GITHUB_KEY"] = "ghp_yourTokenHere"

## Configuration & shared helpers

Set **`INPUT_CSV`** to your file (e.g. `docs/github_users_karachi_full.csv`), then run the next cell once before any enrichment section.

In [4]:
import asyncio
import aiohttp
import csv
import json
import os
import re
import time
from itertools import cycle
from pathlib import Path

# ── Edit these ──────────────────────────────────────────────────────────────
INPUT_CSV = "docs/github_users_islamabad_full.csv"
MAX_CONCURRENT_USERS = 5
CHECKPOINT_SAVE_EVERY = 100     # save CSV + checkpoint every N users
SCRAPE_DELAY_SEC = 1.0          # delay between profile scrapes (section 3)
MAX_REPOS_FOR_METADATA = 30   # cap repos stored in Repo_Metadata JSON
# ───────────────────────────────────────────────────────────────────────────

REST_USER_URL = "https://api.github.com/users/{}"
REST_REPOS_URL = "https://api.github.com/users/{}/repos"
GRAPHQL_URL = "https://api.github.com/graphql"

token_pool = None
token_lock = None
TOKENS: list[str] = []


def load_github_key(env_path=".env") -> str | None:
    """Load GitHub PAT from GITHUB_KEY only (.env file or process env)."""
    key = os.environ.get("GITHUB_KEY", "").strip()
    if key:
        return key
    try:
        with open(env_path, "r", encoding="utf-8-sig") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                name, _, value = line.partition("=")
                if name.strip() == "GITHUB_KEY":
                    value = value.strip().strip('"').strip("'")
                    if value:
                        return value
    except FileNotFoundError:
        pass
    return None


def init_tokens() -> bool:
    global token_pool, token_lock, TOKENS
    candidates = [
        ".env",
        Path.cwd() / ".env",
        Path(INPUT_CSV).resolve().parent.parent / ".env",
    ]
    token = None
    loaded_from = None
    for path in candidates:
        token = load_github_key(str(path))
        if token:
            loaded_from = Path(path).resolve()
            break
    if not token:
        print("No GITHUB_KEY found. Add GITHUB_KEY=ghp_... to .env or set os.environ['GITHUB_KEY'].")
        return False
    TOKENS = [token]
    print(f"Loaded GITHUB_KEY from {loaded_from}")
    token_pool = cycle(TOKENS)
    token_lock = asyncio.Lock()
    return True


async def get_next_token() -> str:
    async with token_lock:
        try:
            return next(token_pool)
        except StopIteration:
            return TOKENS[0]


def api_headers(token: str) -> dict:
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }


async def github_get(session, url: str, retry: int = 0):
    if retry > 5:
        return None, "max_retries"
    token = await get_next_token()
    try:
        async with session.get(
            url,
            headers=api_headers(token),
            timeout=aiohttp.ClientTimeout(total=30),
        ) as resp:
            if resp.status == 200:
                return await resp.json(), None
            if resp.status == 404:
                return None, "not_found"
            if resp.status in (403, 429):
                print("  Rate limited — waiting 10 minutes...")
                await asyncio.sleep(600)
                return await github_get(session, url, retry + 1)
            return None, f"http_{resp.status}"
    except asyncio.TimeoutError:
        await asyncio.sleep(2)
        return await github_get(session, url, retry + 1)
    except Exception:
        return None, "error"


def load_csv_rows() -> tuple[list[dict], list[str]]:
    with open(INPUT_CSV, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fields = list(reader.fieldnames or [])
        rows = list(reader)
    if "Username" not in fields:
        raise ValueError("CSV must have a Username column")
    return rows, fields


def ensure_columns(fields: list[str], columns: list[str]) -> list[str]:
    for col in columns:
        if col not in fields:
            fields.append(col)
    return fields


def write_csv(rows: list[dict], fields: list[str]) -> None:
    with open(INPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def load_checkpoint(path: str) -> dict:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"Checkpoint {path}: {len(data):,} users cached")
        return data
    return {}


def save_checkpoint(path: str, data: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f)


def api_row_is_filled(row: dict) -> bool:
    """True when unified API columns already have real data."""
    v = str(row.get("Public_Repositories", "")).strip()
    return v not in ("", "N/A")


async def paginate_user_repos(session, username: str) -> list[dict]:
    """All public repos for a user (paginated REST)."""
    repos: list[dict] = []
    page = 1
    while True:
        url = (
            f"{REST_REPOS_URL.format(username.strip())}"
            f"?per_page=100&page={page}&type=owner&sort=updated"
        )
        data, err = await github_get(session, url)
        if err or data is None:
            break
        if not isinstance(data, list) or not data:
            break
        repos.extend(data)
        if len(data) < 100:
            break
        page += 1
    return repos


print(f"Input CSV: {Path(INPUT_CSV).resolve()}")

Input CSV: E:\git-city\docs\github_users_islamabad_full.csv


## 1. Unified GraphQL — all API columns (recommended)

One query per page of repos returns **followers**, **public repo count**, **lifetime commits**, **total stars**, **repo names**, and **repo metadata** together.

Checkpoint: `checkpoint_unified.json`

In [5]:
CHECKPOINT_UNIFIED = "checkpoint_unified.json"

UNIFIED_COLUMNS = [
    "Public_Repositories",
    "Lifetime_Commits",
    "Followers",
    "Total_Stars",
    "Repo_Names",
    "Repo_Metadata",
]

# One GraphQL request per 100 repos (vs 5+ REST/GraphQL round-trips in legacy sections)
UNIFIED_USER_QUERY = """
query($login: String!, $cursor: String) {
  user(login: $login) {
    followers { totalCount }
    publicRepos: repositories(ownerAffiliations: OWNER, privacy: PUBLIC) {
      totalCount
    }
    ownedRepos: repositories(
      first: 100
      after: $cursor
      ownerAffiliations: OWNER
      isFork: false
      orderBy: { field: UPDATED_AT, direction: DESC }
    ) {
      pageInfo { hasNextPage endCursor }
      nodes {
        name
        description
        stargazerCount
        forkCount
        issues { totalCount }
        createdAt
        updatedAt
        primaryLanguage { name }
        repositoryTopics(first: 10) { nodes { topic { name } } }
        licenseInfo { spdxId }
        isArchived
        isFork
        defaultBranchRef {
          target {
            ... on Commit {
              history { totalCount }
            }
          }
        }
      }
    }
  }
}
"""


async def github_graphql(
    session: aiohttp.ClientSession,
    query: str,
    variables: dict,
    retry: int = 0,
) -> dict | None:
    if retry > 5:
        return None
    token = await get_next_token()
    headers = {"Authorization": f"bearer {token}", "Content-Type": "application/json"}
    try:
        async with session.post(
            GRAPHQL_URL,
            json={"query": query, "variables": variables},
            headers=headers,
            timeout=aiohttp.ClientTimeout(total=90),
        ) as resp:
            if resp.status == 401:
                print("  GraphQL 401 Bad credentials — fix GITHUB_KEY in .env")
                return None
            if resp.status in (403, 429):
                print("  GraphQL rate limited — waiting 10 minutes...")
                await asyncio.sleep(600)
                return await github_graphql(session, query, variables, retry + 1)
            if resp.status != 200:
                print(f"  GraphQL HTTP {resp.status}")
                return None
            body = await resp.json()
            if body.get("errors"):
                msg = body["errors"][0].get("message", body["errors"])
                if retry < 5 and ("rate limit" in str(msg).lower() or "secondary" in str(msg).lower()):
                    print(f"  GraphQL error — waiting 60s: {msg}")
                    await asyncio.sleep(60)
                    return await github_graphql(session, query, variables, retry + 1)
                print(f"  GraphQL error: {msg}")
                return None
            return body.get("data")
    except Exception:
        return None


def gql_node_to_metadata(node: dict) -> dict:
    topics = [
        n["topic"]["name"]
        for n in (node.get("repositoryTopics") or {}).get("nodes", [])
        if n.get("topic")
    ]
    return {
        "name": node.get("name"),
        "language": (node.get("primaryLanguage") or {}).get("name"),
        "stars": node.get("stargazerCount", 0),
        "forks": node.get("forkCount", 0),
        "open_issues": (node.get("issues") or {}).get("totalCount", 0),
        "created_at": node.get("createdAt"),
        "updated_at": node.get("updatedAt"),
        "description": (node.get("description") or "")[:500],
        "topics": topics,
        "license": (node.get("licenseInfo") or {}).get("spdxId"),
        "archived": node.get("isArchived", False),
        "fork": node.get("isFork", False),
    }


async def fetch_user_bundle(session: aiohttp.ClientSession, username: str) -> dict | None:
    """All API fields for one user in as few GraphQL calls as possible."""
    login = username.strip()
    cursor = None
    followers = None
    public_repos = None
    lifetime_commits = 0
    total_stars = 0
    names: list[str] = []
    meta_nodes: list[dict] = []

    while True:
        data = await github_graphql(
            session,
            UNIFIED_USER_QUERY,
            {"login": login, "cursor": cursor},
        )
        user = (data or {}).get("user")
        if not user:
            return None

        if followers is None:
            followers = (user.get("followers") or {}).get("totalCount", 0)
            public_repos = (user.get("publicRepos") or {}).get("totalCount", 0)

        conn = user.get("ownedRepos") or {}
        for node in conn.get("nodes") or []:
            names.append(node.get("name") or "")
            total_stars += int(node.get("stargazerCount") or 0)
            meta_nodes.append(node)
            ref = node.get("defaultBranchRef")
            if ref and ref.get("target"):
                lifetime_commits += ref["target"]["history"]["totalCount"]

        page = conn.get("pageInfo") or {}
        if page.get("hasNextPage"):
            cursor = page.get("endCursor")
        else:
            break

    names = [n for n in names if n]
    metadata = [gql_node_to_metadata(n) for n in meta_nodes[:MAX_REPOS_FOR_METADATA]]

    return {
        "Public_Repositories": public_repos,
        "Lifetime_Commits": lifetime_commits,
        "Followers": followers,
        "Total_Stars": total_stars,
        "Repo_Names": "|".join(names),
        "Repo_Metadata": json.dumps(metadata, ensure_ascii=False),
    }


async def run_unified_api_update():
    if not init_tokens():
        return
    rows, fields = load_csv_rows()
    fields = ensure_columns(fields, UNIFIED_COLUMNS)
    checkpoint = load_checkpoint(CHECKPOINT_UNIFIED)
    cp_lock = asyncio.Lock()

    already_ok = 0
    pending = []
    for row in rows:
        u = row["Username"].strip()
        if u in checkpoint and api_row_is_filled(checkpoint[u]):
            for col in UNIFIED_COLUMNS:
                row[col] = checkpoint[u][col]
            already_ok += 1
        elif api_row_is_filled(row):
            already_ok += 1
        else:
            pending.append(row)

    print(
        f"Unified API: {len(rows):,} users — "
        f"{already_ok:,} already filled, {len(pending):,} to fetch"
    )
    print("(~1 GraphQL call per 100 owned non-fork repos per user)\n")

    if not pending:
        write_csv(rows, fields)
        print("Nothing to do.")
        return

    sem = asyncio.Semaphore(MAX_CONCURRENT_USERS)
    start = time.time()
    done = 0

    async def one(session, row):
        nonlocal done
        async with sem:
            u = row["Username"].strip()
            bundle = await fetch_user_bundle(session, u)
            if bundle is None:
                # Do not write N/A — leave row for retry on next run
                return row
            for col in UNIFIED_COLUMNS:
                row[col] = bundle[col]
            async with cp_lock:
                checkpoint[u] = {col: row[col] for col in UNIFIED_COLUMNS}
            done += 1
            if done % CHECKPOINT_SAVE_EVERY == 0:
                save_checkpoint(CHECKPOINT_UNIFIED, checkpoint)
                write_csv(rows, fields)
                rate = done / (time.time() - start)
                eta = (len(pending) - done) / rate / 60 if rate else 0
                print(
                    f"  [{done:,}/{len(pending):,}] {rate:.1f}/s | ETA {eta:.1f} min | "
                    f"checkpoint + CSV saved"
                )
            return row

    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit=200)) as session:
        await asyncio.gather(*[asyncio.create_task(one(session, r)) for r in pending])

    save_checkpoint(CHECKPOINT_UNIFIED, checkpoint)
    write_csv(rows, fields)

    still_missing = sum(1 for r in rows if not api_row_is_filled(r))
    if still_missing == 0:
        if os.path.exists(CHECKPOINT_UNIFIED):
            os.remove(CHECKPOINT_UNIFIED)
        print(f"\nDone — all {len(rows):,} users filled in {INPUT_CSV}")
    else:
        print(
            f"\nPartial run — {len(rows) - still_missing:,} filled, "
            f"{still_missing:,} still empty/N/A."
        )
        print(f"Checkpoint kept: {CHECKPOINT_UNIFIED} — re-run this cell to continue.")
        print(f"CSV updated: {INPUT_CSV}")


await run_unified_api_update()

Loaded GITHUB_KEY from E:\git-city\.env
Unified API: 19,454 users — 0 cached, 19,454 to fetch
(~1 GraphQL call per 100 owned non-fork repos per user)

  [100/19,454] 3.1/s | ETA 105.5 min | checkpoint saved
  [200/19,454] 3.2/s | ETA 101.1 min | checkpoint saved
  [300/19,454] 3.6/s | ETA 88.9 min | checkpoint saved
  [400/19,454] 3.6/s | ETA 89.2 min | checkpoint saved
  [500/19,454] 3.7/s | ETA 85.4 min | checkpoint saved
  [600/19,454] 3.7/s | ETA 84.4 min | checkpoint saved
  [700/19,454] 3.6/s | ETA 85.8 min | checkpoint saved
  [800/19,454] 3.7/s | ETA 84.6 min | checkpoint saved
  [900/19,454] 3.7/s | ETA 84.0 min | checkpoint saved
  [1,000/19,454] 3.7/s | ETA 83.5 min | checkpoint saved
  [1,100/19,454] 3.7/s | ETA 83.4 min | checkpoint saved
  [1,200/19,454] 3.7/s | ETA 82.0 min | checkpoint saved
  [1,300/19,454] 3.7/s | ETA 82.0 min | checkpoint saved
  [1,400/19,454] 3.7/s | ETA 82.0 min | checkpoint saved
  [1,500/19,454] 3.7/s | ETA 81.7 min | checkpoint saved
  [1,600/1

## 1 legacy. Commits + repos only (skip if you ran unified above)

Uses **REST** + separate **GraphQL** (2+ calls per user). Prefer **section 1 unified** instead.

Checkpoint: `checkpoint_commits.json`

In [ ]:
CHECKPOINT_COMMITS = "checkpoint_commits.json"
COLUMNS_COMMITS = ["Public_Repositories", "Lifetime_Commits"]

LIFETIME_COMMITS_QUERY = """
query($login: String!, $cursor: String) {
  user(login: $login) {
    repositories(
      first: 100
      isFork: false
      ownerAffiliations: OWNER
      after: $cursor
    ) {
      pageInfo { hasNextPage endCursor }
      nodes {
        defaultBranchRef {
          target {
            ... on Commit {
              history { totalCount }
            }
          }
        }
      }
    }
  }
}
"""


async def fetch_public_repos(session, username: str) -> str | int:
    data, err = await github_get(session, REST_USER_URL.format(username.strip()))
    if err == "not_found":
        return "Not Found"
    if not data:
        return "N/A"
    return data.get("public_repos", 0)


async def fetch_lifetime_commits(session, username: str, retry: int = 0) -> int:
    if retry > 5:
        return -1
    total = 0
    cursor = None
    token = await get_next_token()
    headers = {"Authorization": f"bearer {token}", "Content-Type": "application/json"}
    while True:
        payload = {"query": LIFETIME_COMMITS_QUERY, "variables": {"login": username, "cursor": cursor}}
        try:
            async with session.post(
                GRAPHQL_URL,
                json=payload,
                headers=headers,
                timeout=aiohttp.ClientTimeout(total=60),
            ) as resp:
                if resp.status in (403, 429):
                    print("  GraphQL rate limited — waiting 10 minutes...")
                    await asyncio.sleep(600)
                    return await fetch_lifetime_commits(session, username, retry + 1)
                if resp.status != 200:
                    return -1
                body = await resp.json()
                if body.get("errors") or not body.get("data", {}).get("user"):
                    return -1
                repos = body["data"]["user"]["repositories"]
                for node in repos["nodes"]:
                    ref = node.get("defaultBranchRef")
                    if ref and ref.get("target"):
                        total += ref["target"]["history"]["totalCount"]
                if repos["pageInfo"]["hasNextPage"]:
                    cursor = repos["pageInfo"]["endCursor"]
                else:
                    break
        except Exception:
            return -1
    return total


async def run_commits_update():
    if not init_tokens():
        return
    rows, fields = load_csv_rows()
    fields = ensure_columns(fields, COLUMNS_COMMITS)
    checkpoint = load_checkpoint(CHECKPOINT_COMMITS)
    cp_lock = asyncio.Lock()

    pending = []
    for row in rows:
        u = row["Username"].strip()
        if u in checkpoint:
            row["Public_Repositories"] = checkpoint[u]["Public_Repositories"]
            row["Lifetime_Commits"] = checkpoint[u]["Lifetime_Commits"]
        else:
            pending.append(row)

    print(f"{len(rows):,} users — {len(rows) - len(pending):,} cached, {len(pending):,} to fetch\n")
    if not pending:
        write_csv(rows, fields)
        print("Nothing to do.")
        return

    sem = asyncio.Semaphore(MAX_CONCURRENT_USERS)
    start = time.time()
    done = 0
    save_every = 100

    async def one(session, row):
        nonlocal done
        async with sem:
            u = row["Username"].strip()
            repos, commits = await asyncio.gather(
                fetch_public_repos(session, u),
                fetch_lifetime_commits(session, u),
            )
            row["Public_Repositories"] = repos
            row["Lifetime_Commits"] = commits if commits >= 0 else "N/A"
            async with cp_lock:
                checkpoint[u] = {
                    "Public_Repositories": row["Public_Repositories"],
                    "Lifetime_Commits": row["Lifetime_Commits"],
                }
            done += 1
            if done % save_every == 0:
                save_checkpoint(CHECKPOINT_COMMITS, checkpoint)
                elapsed = time.time() - start
                rate = done / elapsed
                eta = (len(pending) - done) / rate / 60 if rate else 0
                print(f"  [{done:,}/{len(pending):,}] {rate:.1f}/s | ETA {eta:.1f} min | checkpoint saved")
            return row

    connector = aiohttp.TCPConnector(limit=200)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [asyncio.create_task(one(session, r)) for r in pending]
        await asyncio.gather(*tasks)

    save_checkpoint(CHECKPOINT_COMMITS, checkpoint)
    write_csv(rows, fields)
    if os.path.exists(CHECKPOINT_COMMITS):
        os.remove(CHECKPOINT_COMMITS)
    print(f"\nDone — updated {INPUT_CSV}")


await run_commits_update()

Loaded 10 token(s) from E:\git-city\.env
19,454 users — 0 cached, 19,454 to fetch

  [100/19,454] 9.6/s | ETA 33.7 min | checkpoint saved
  [200/19,454] 10.2/s | ETA 31.4 min | checkpoint saved
  [300/19,454] 10.4/s | ETA 30.7 min | checkpoint saved
  [400/19,454] 10.4/s | ETA 30.6 min | checkpoint saved
  [500/19,454] 10.3/s | ETA 30.7 min | checkpoint saved


CancelledError: 

## 2a legacy. Followers only (skip if you ran unified)

Duplicate of part of the unified query. Checkpoint: `checkpoint_followers.json`.

In [ ]:
CHECKPOINT_FOLLOWERS = "checkpoint_followers.json"
COLUMN_FOLLOWERS = "Followers"


async def run_followers_update():
    if not init_tokens():
        return
    rows, fields = load_csv_rows()
    fields = ensure_columns(fields, [COLUMN_FOLLOWERS])
    checkpoint = load_checkpoint(CHECKPOINT_FOLLOWERS)
    cp_lock = asyncio.Lock()
    pending = [r for r in rows if r["Username"].strip() not in checkpoint]
    for r in rows:
        u = r["Username"].strip()
        if u in checkpoint:
            r[COLUMN_FOLLOWERS] = checkpoint[u][COLUMN_FOLLOWERS]

    print(f"Followers: {len(pending):,} users to fetch")
    sem = asyncio.Semaphore(MAX_CONCURRENT_USERS)
    done = 0

    async def one(session, row):
        nonlocal done
        async with sem:
            u = row["Username"].strip()
            data, err = await github_get(session, REST_USER_URL.format(u))
            if err == "not_found":
                val = "Not Found"
            elif data:
                val = data.get("followers", 0)
            else:
                val = "N/A"
            row[COLUMN_FOLLOWERS] = val
            async with cp_lock:
                checkpoint[u] = {COLUMN_FOLLOWERS: val}
            done += 1
            if done % 100 == 0:
                save_checkpoint(CHECKPOINT_FOLLOWERS, checkpoint)
                print(f"  [{done:,}/{len(pending):,}] checkpoint saved")

    if pending:
        async with aiohttp.ClientSession() as session:
            await asyncio.gather(*[asyncio.create_task(one(session, r)) for r in pending])
        save_checkpoint(CHECKPOINT_FOLLOWERS, checkpoint)

    write_csv(rows, fields)
    if os.path.exists(CHECKPOINT_FOLLOWERS):
        os.remove(CHECKPOINT_FOLLOWERS)
    print(f"Done — {COLUMN_FOLLOWERS} written to {INPUT_CSV}")


await run_followers_update()

## 2b legacy. Total stars only (skip if you ran unified)

In [ ]:
CHECKPOINT_STARS = "checkpoint_stars.json"
COLUMN_STARS = "Total_Stars"


async def run_stars_update():
    if not init_tokens():
        return
    rows, fields = load_csv_rows()
    fields = ensure_columns(fields, [COLUMN_STARS])
    checkpoint = load_checkpoint(CHECKPOINT_STARS)
    cp_lock = asyncio.Lock()
    pending = [r for r in rows if r["Username"].strip() not in checkpoint]
    for r in rows:
        u = r["Username"].strip()
        if u in checkpoint:
            r[COLUMN_STARS] = checkpoint[u][COLUMN_STARS]

    print(f"Total stars: {len(pending):,} users to fetch")
    sem = asyncio.Semaphore(MAX_CONCURRENT_USERS)
    done = 0

    async def one(session, row):
        nonlocal done
        async with sem:
            u = row["Username"].strip()
            repos = await paginate_user_repos(session, u)
            if repos:
                val = sum(int(r.get("stargazers_count") or 0) for r in repos)
            else:
                val = "N/A"
            row[COLUMN_STARS] = val
            async with cp_lock:
                checkpoint[u] = {COLUMN_STARS: val}
            done += 1
            if done % 50 == 0:
                save_checkpoint(CHECKPOINT_STARS, checkpoint)
                print(f"  [{done:,}/{len(pending):,}] checkpoint saved")

    if pending:
        async with aiohttp.ClientSession() as session:
            await asyncio.gather(*[asyncio.create_task(one(session, r)) for r in pending])
        save_checkpoint(CHECKPOINT_STARS, checkpoint)

    write_csv(rows, fields)
    if os.path.exists(CHECKPOINT_STARS):
        os.remove(CHECKPOINT_STARS)
    print(f"Done — {COLUMN_STARS} written to {INPUT_CSV}")


await run_stars_update()

## 2c. API — Repository names

Pipe-separated public repo names (owner repos, sorted by last update). Column: **`Repo_Names`**.

In [ ]:
CHECKPOINT_REPO_NAMES = "checkpoint_repo_names.json"
COLUMN_REPO_NAMES = "Repo_Names"


async def run_repo_names_update():
    if not init_tokens():
        return
    rows, fields = load_csv_rows()
    fields = ensure_columns(fields, [COLUMN_REPO_NAMES])
    checkpoint = load_checkpoint(CHECKPOINT_REPO_NAMES)
    cp_lock = asyncio.Lock()
    pending = [r for r in rows if r["Username"].strip() not in checkpoint]
    for r in rows:
        u = r["Username"].strip()
        if u in checkpoint:
            r[COLUMN_REPO_NAMES] = checkpoint[u][COLUMN_REPO_NAMES]

    print(f"Repo names: {len(pending):,} users to fetch")
    sem = asyncio.Semaphore(MAX_CONCURRENT_USERS)
    done = 0

    async def one(session, row):
        nonlocal done
        async with sem:
            u = row["Username"].strip()
            repos = await paginate_user_repos(session, u)
            names = [r.get("name", "") for r in repos if r.get("name")]
            val = "|".join(names) if names else ""
            row[COLUMN_REPO_NAMES] = val
            async with cp_lock:
                checkpoint[u] = {COLUMN_REPO_NAMES: val}
            done += 1
            if done % 50 == 0:
                save_checkpoint(CHECKPOINT_REPO_NAMES, checkpoint)
                print(f"  [{done:,}/{len(pending):,}] checkpoint saved")

    if pending:
        async with aiohttp.ClientSession() as session:
            await asyncio.gather(*[asyncio.create_task(one(session, r)) for r in pending])
        save_checkpoint(CHECKPOINT_REPO_NAMES, checkpoint)

    write_csv(rows, fields)
    if os.path.exists(CHECKPOINT_REPO_NAMES):
        os.remove(CHECKPOINT_REPO_NAMES)
    print(f"Done — {COLUMN_REPO_NAMES} written to {INPUT_CSV}")


await run_repo_names_update()

## 2d legacy. Repo metadata only (skip if you ran unified)

In [ ]:
CHECKPOINT_REPO_META = "checkpoint_repo_metadata.json"
COLUMN_REPO_META = "Repo_Metadata"


def repo_to_metadata(r: dict) -> dict:
    lic = r.get("license")
    return {
        "name": r.get("name"),
        "language": r.get("language"),
        "stars": r.get("stargazers_count", 0),
        "forks": r.get("forks_count", 0),
        "open_issues": r.get("open_issues_count", 0),
        "created_at": r.get("created_at"),
        "updated_at": r.get("updated_at"),
        "description": (r.get("description") or "")[:500],
        "topics": r.get("topics") or [],
        "license": lic.get("spdx_id") if isinstance(lic, dict) else None,
        "archived": r.get("archived", False),
        "fork": r.get("fork", False),
    }


async def run_repo_metadata_update():
    if not init_tokens():
        return
    rows, fields = load_csv_rows()
    fields = ensure_columns(fields, [COLUMN_REPO_META])
    checkpoint = load_checkpoint(CHECKPOINT_REPO_META)
    cp_lock = asyncio.Lock()
    pending = [r for r in rows if r["Username"].strip() not in checkpoint]
    for r in rows:
        u = r["Username"].strip()
        if u in checkpoint:
            r[COLUMN_REPO_META] = checkpoint[u][COLUMN_REPO_META]

    print(f"Repo metadata: {len(pending):,} users to fetch")
    sem = asyncio.Semaphore(MAX_CONCURRENT_USERS)
    done = 0

    async def one(session, row):
        nonlocal done
        async with sem:
            u = row["Username"].strip()
            repos = await paginate_user_repos(session, u)
            meta = [repo_to_metadata(r) for r in repos[:MAX_REPOS_FOR_METADATA]]
            val = json.dumps(meta, ensure_ascii=False) if meta is not None else "N/A"
            row[COLUMN_REPO_META] = val
            async with cp_lock:
                checkpoint[u] = {COLUMN_REPO_META: val}
            done += 1
            if done % 50 == 0:
                save_checkpoint(CHECKPOINT_REPO_META, checkpoint)
                print(f"  [{done:,}/{len(pending):,}] checkpoint saved")

    if pending:
        async with aiohttp.ClientSession() as session:
            await asyncio.gather(*[asyncio.create_task(one(session, r)) for r in pending])
        save_checkpoint(CHECKPOINT_REPO_META, checkpoint)

    write_csv(rows, fields)
    if os.path.exists(CHECKPOINT_REPO_META):
        os.remove(CHECKPOINT_REPO_META)
    print(f"Done — {COLUMN_REPO_META} written to {INPUT_CSV}")


await run_repo_metadata_update()

## 3. Web scraping — Contribution heatmap

Fetches the public profile page and parses:
- **`Contributions_Last_Year`** — total from the "N contributions in the last year" heading
- **`Heatmap_Data`** — JSON list of `{date, level, count}` from `data-date` / `data-level` / `data-count` on calendar cells

No GitHub token required. Uses **`SCRAPE_DELAY_SEC`** between requests. Checkpoint: `checkpoint_heatmap.json`.

Install if needed: `pip install beautifulsoup4 lxml`

In [3]:
try:
    from bs4 import BeautifulSoup
except ImportError:
    raise ImportError("Run: pip install beautifulsoup4 lxml")

CHECKPOINT_HEATMAP = "checkpoint_heatmap.json"
COLUMNS_HEATMAP = ["Contributions_Last_Year", "Heatmap_Data"]
PROFILE_URL = "https://github.com/{}"
SCRAPE_HEADERS = {
    "User-Agent": "git-city-data-pipeline/1.0 (research; contact via repo)",
    "Accept": "text/html",
}

CONTRIBUTIONS_RE = re.compile(
    r"([\d,]+)\s+contributions?\s+in the last year", re.IGNORECASE
)


def parse_contribution_page(html: str) -> tuple[str | int, str]:
    soup = BeautifulSoup(html, "lxml")
    total = "N/A"
    text = soup.get_text(" ", strip=True)
    m = CONTRIBUTIONS_RE.search(text)
    if m:
        total = int(m.group(1).replace(",", ""))

    days = []
    for td in soup.select("td.ContributionCalendar-day, tool-tip[data-date]"):
        date = td.get("data-date")
        if not date:
            continue
        level = int(td.get("data-level") or 0)
        count_raw = td.get("data-count")
        count = int(count_raw) if count_raw is not None else 0
        days.append({"date": date, "level": level, "count": count})

    if not days:
        for el in soup.select("[data-date][data-level]"):
            date = el.get("data-date")
            if not date:
                continue
            days.append({
                "date": date,
                "level": int(el.get("data-level") or 0),
                "count": int(el.get("data-count") or 0),
            })

    heatmap_json = json.dumps(days, ensure_ascii=False)
    return total, heatmap_json


CONTRIBUTIONS_URL = "https://github.com/users/{}/contributions"


async def fetch_html(session: aiohttp.ClientSession, url: str) -> str | None:
    try:
        async with session.get(
            url,
            headers=SCRAPE_HEADERS,
            timeout=aiohttp.ClientTimeout(total=30),
        ) as resp:
            if resp.status == 404:
                return None
            if resp.status != 200:
                return ""
            return await resp.text()
    except Exception:
        return ""


async def fetch_profile_html(session: aiohttp.ClientSession, username: str) -> str | None:
    u = username.strip()
    html = await fetch_html(session, PROFILE_URL.format(u))
    if html is None:
        return None
    if html and "ContributionCalendar-day" not in html and "data-date" not in html:
        fragment = await fetch_html(session, CONTRIBUTIONS_URL.format(u))
        if fragment:
            html = fragment
    return html


async def run_heatmap_update():
    rows, fields = load_csv_rows()
    fields = ensure_columns(fields, COLUMNS_HEATMAP)
    checkpoint = load_checkpoint(CHECKPOINT_HEATMAP)

    pending = [r for r in rows if r["Username"].strip() not in checkpoint]
    for r in rows:
        u = r["Username"].strip()
        if u in checkpoint:
            r["Contributions_Last_Year"] = checkpoint[u]["Contributions_Last_Year"]
            r["Heatmap_Data"] = checkpoint[u]["Heatmap_Data"]

    print(f"Heatmap scrape: {len(pending):,} users (delay {SCRAPE_DELAY_SEC}s)\n")
    done = 0

    async with aiohttp.ClientSession() as session:
        for row in pending:
            u = row["Username"].strip()
            html = await fetch_profile_html(session, u)
            if html is None:
                total, heat = "Not Found", "[]"
            elif not html:
                total, heat = "N/A", "[]"
            else:
                total, heat = parse_contribution_page(html)
            row["Contributions_Last_Year"] = total
            row["Heatmap_Data"] = heat
            checkpoint[u] = {
                "Contributions_Last_Year": total,
                "Heatmap_Data": heat,
            }
            done += 1
            if done % 50 == 0:
                save_checkpoint(CHECKPOINT_HEATMAP, checkpoint)
                print(f"  [{done:,}/{len(pending):,}] checkpoint saved")
            await asyncio.sleep(SCRAPE_DELAY_SEC)

    if pending:
        save_checkpoint(CHECKPOINT_HEATMAP, checkpoint)

    write_csv(rows, fields)
    if os.path.exists(CHECKPOINT_HEATMAP):
        os.remove(CHECKPOINT_HEATMAP)
    print(f"\nDone — heatmap columns written to {INPUT_CSV}")


await run_heatmap_update()

Heatmap scrape: 19,454 users (delay 1.0s)

  [50/19,454] checkpoint saved
  [100/19,454] checkpoint saved
  [150/19,454] checkpoint saved
  [200/19,454] checkpoint saved
  [250/19,454] checkpoint saved
  [300/19,454] checkpoint saved
  [350/19,454] checkpoint saved
  [400/19,454] checkpoint saved
  [450/19,454] checkpoint saved
  [500/19,454] checkpoint saved
  [550/19,454] checkpoint saved
  [600/19,454] checkpoint saved
  [650/19,454] checkpoint saved
  [700/19,454] checkpoint saved
  [750/19,454] checkpoint saved
  [800/19,454] checkpoint saved
  [850/19,454] checkpoint saved
  [900/19,454] checkpoint saved
  [950/19,454] checkpoint saved
  [1,000/19,454] checkpoint saved
  [1,050/19,454] checkpoint saved
  [1,100/19,454] checkpoint saved
  [1,150/19,454] checkpoint saved
  [1,200/19,454] checkpoint saved
  [1,250/19,454] checkpoint saved
  [1,300/19,454] checkpoint saved
  [1,350/19,454] checkpoint saved
  [1,400/19,454] checkpoint saved
  [1,450/19,454] checkpoint saved
  [1,500/1

CancelledError: 